In [5]:
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("Is CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA version: 12.1
Is CUDA available: True
GPU Name: NVIDIA GeForce RTX 3070 Ti Laptop GPU


In [6]:
%pip install transformers scikit-learn pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [13]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from tqdm import tqdm

In [14]:
def get_device():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    return device

In [15]:
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df["text"].tolist(),
        df["label"].tolist(),
        test_size=0.1,
        random_state=42
    )
    return train_texts, val_texts, train_labels, val_labels

In [16]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=256
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [17]:
def load_model(device):
    model_name = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    model.to(device)
    return tokenizer, model

In [18]:
def create_dataloaders(train_texts, val_texts, train_labels, val_labels, tokenizer):
    train_dataset = IMDBDataset(train_texts, train_labels, tokenizer)
    val_dataset = IMDBDataset(val_texts, val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16)

    return train_loader, val_loader

In [19]:
def train_model(model, train_loader, device, epochs=3):
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"\nEpoch {epoch+1} Training Loss: {avg_loss}")

In [20]:
def evaluate_model(model, val_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=1)

            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    accuracy = correct / total
    print("Validation Accuracy:", accuracy)

In [23]:
def save_model(model, tokenizer):
    model.save_pretrained("imdb_bert_model")
    tokenizer.save_pretrained("imdb_bert_model")
    print("Model saved successfully.")

In [24]:
def main():
    device = get_device()

    train_texts, val_texts, train_labels, val_labels = load_data("imdb_train.csv")

    tokenizer, model = load_model(device)

    train_loader, val_loader = create_dataloaders(
        train_texts, val_texts, train_labels, val_labels, tokenizer
    )

    train_model(model, train_loader, device, epochs=3)

    evaluate_model(model, val_loader, device)

    save_model(model, tokenizer)

In [25]:
main()

Using device: cuda
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 699.15it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa


Epoch 1 Training Loss: 0.2694448465256347


100%|██████████| 1407/1407 [09:52<00:00,  2.37it/s]



Epoch 2 Training Loss: 0.14364338912600658


100%|██████████| 1407/1407 [09:49<00:00,  2.39it/s]



Epoch 3 Training Loss: 0.07108762974793909
Validation Accuracy: 0.9164


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]


Model saved successfully.
